In [19]:
import pandas as pd
import numpy as np
import warnings, re
import statsmodels.api as sm
warnings.filterwarnings('ignore')

In [20]:
df = pd.read_pickle('../data/cleaned_sumo_bouts.pkl')
df

,basho,day,bout_id,technique,win,name,heya,shusshin,birth_date,age,...,cum_win_rate,wins_prev6,bouts_prev6,win_rate_prev6,opponent_cum_wins,opponent_cum_bouts,opponent_cum_win_rate,opponent_wins_prev6,opponent_bouts_prev6,opponent_win_rate_prev6
16,2015.05,2,45227,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,216,438,0.493151,40.0,77.0,0.519481
17,2015.05,3,45264,sukuinage,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.529412,7.0,15.0,0.466667,83,165,0.503030,40.0,83.0,0.481928
18,2015.05,4,45295,tsukiotoshi,0,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,3,6,0.500000,2.0,3.0,0.666667
19,2015.05,5,45329,uwatenage,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.473684,7.0,15.0,0.466667,11,26,0.423077,8.0,22.0,0.363636
20,2015.05,6,45365,hatakikomi,1,Abi,Shikoroyama,Saitama,1994-05-04,21.0,...,0.500000,7.0,15.0,0.466667,117,266,0.439850,33.0,76.0,0.434211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152131,2022.11,11,67538,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.465696,38.0,90.0,0.422222,208,409,0.508557,39.0,76.0,0.513158
152132,2022.11,12,67572,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.464730,38.0,90.0,0.422222,302,599,0.504174,42.0,88.0,0.477273
152133,2022.11,13,67605,yorikiri,1,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.463768,38.0,90.0,0.422222,472,977,0.483112,43.0,90.0,0.477778
152134,2022.11,14,67640,yorikiri,0,Yutakayama,Tokitsukaze,Niigata,1993-09-22,29.1,...,0.464876,38.0,90.0,0.422222,33,71,0.464789,27.0,58.0,0.465517


In [21]:
df.columns

Index(['basho', 'day', 'bout_id', 'technique', 'win', 'name', 'heya',
       'shusshin', 'birth_date', 'age', 'division', 'height', 'weight',
       'experience', 'rank', 'rank_num', 'opponent_name', 'opponent_heya',
       'opponent_shusshin', 'opponent_age', 'opponent_division',
       'opponent_height', 'opponent_weight', 'opponent_experience',
       'opponent_rank', 'opponent_rank_num', 'age_diff', 'height_diff',
       'weight_diff', 'experience_diff', 'rank_diff', 'flag_higher_division',
       'flag_lower_division', 'flag_same_hometown', 'flag_first_day',
       'flag_last_day', 'cum_wins', 'cum_bouts', 'cum_win_rate', 'wins_prev6',
       'bouts_prev6', 'win_rate_prev6', 'opponent_cum_wins',
       'opponent_cum_bouts', 'opponent_cum_win_rate', 'opponent_wins_prev6',
       'opponent_bouts_prev6', 'opponent_win_rate_prev6'],
      dtype='object')

In [22]:
y = df['win']
X = df[[
    'height', 'weight', 'experience', 'rank_num', 'division', 
    'age_diff', 'height_diff','weight_diff', 'experience_diff', 'rank_diff', 
    'flag_higher_division','flag_lower_division','flag_same_hometown', 
    'flag_first_day','flag_last_day', 
    'cum_wins', 'cum_bouts', 'cum_win_rate', 'wins_prev6',
    'bouts_prev6', 'win_rate_prev6', 'opponent_cum_wins',
    'opponent_cum_bouts', 'opponent_cum_win_rate', 'opponent_wins_prev6',
    'opponent_bouts_prev6', 'opponent_win_rate_prev6'
]]

X=pd.get_dummies(X, columns=['division'], drop_first=True)
X

,height,weight,experience,rank_num,age_diff,height_diff,weight_diff,experience_diff,rank_diff,flag_higher_division,...,bouts_prev6,win_rate_prev6,opponent_cum_wins,opponent_cum_bouts,opponent_cum_win_rate,opponent_wins_prev6,opponent_bouts_prev6,opponent_win_rate_prev6,division_Maegashira,division_Makushita
16,185.0,121.0,2.000000,40.0,-9.20,7.0,-19.8,-6.172603,-1.0,0,...,15.0,0.466667,216,438,0.493151,40.0,77.0,0.519481,False,False
17,185.0,121.0,2.000000,40.0,-2.11,7.0,-28.6,-3.169863,3.0,0,...,15.0,0.466667,83,165,0.503030,40.0,83.0,0.481928,False,False
18,185.0,121.0,2.000000,40.0,-3.80,0.5,-28.5,-7.172603,0.0,0,...,15.0,0.466667,3,6,0.500000,2.0,3.0,0.666667,False,False
19,185.0,121.0,2.000000,40.0,-9.70,4.0,-19.0,-13.175342,-1.0,0,...,15.0,0.466667,11,26,0.423077,8.0,22.0,0.363636,False,False
20,185.0,121.0,2.000000,40.0,-8.90,2.0,-51.6,-5.334247,2.0,0,...,15.0,0.466667,117,266,0.439850,33.0,76.0,0.434211,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152131,185.0,178.0,6.673973,31.0,0.50,-5.0,-20.0,1.000000,1.0,0,...,90.0,0.422222,208,409,0.508557,39.0,76.0,0.513158,False,False
152132,185.0,178.0,6.673973,31.0,-2.20,1.0,-22.0,-2.164384,1.0,0,...,90.0,0.422222,302,599,0.504174,42.0,88.0,0.477273,False,False
152133,185.0,178.0,6.673973,31.0,-7.10,2.0,-11.0,-7.167123,-8.0,0,...,90.0,0.422222,472,977,0.483112,43.0,90.0,0.477778,False,False
152134,185.0,178.0,6.673973,31.0,2.60,10.0,19.0,-4.002740,-9.0,0,...,90.0,0.422222,33,71,0.464789,27.0,58.0,0.465517,False,False


In [ ]:
# Try a logit model
